# 夜盘信号稳定超额研究（ES/NQ 期货隔夜 → SOXL regular session）

新采集夜盘数据：股指期货 ES/NQ 连续交易，隔夜（前日18:00 ET → 当日09:30 ET）走势在美股开盘前已知。
用它预测当日 09:30–16:00 SOXL 相对 SOXX 的超额。严格因果对齐、无收益预测、无拟合参数。


## 1. 预登记计划


In [1]:
import sys, hashlib, json
from pathlib import Path

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
CARD = ROOT / 'research' / 'night_session_signal_20261010'
plan_bytes = (CARD / 'plan.json').read_bytes()
plan = json.loads(plan_bytes)
print('plan sha256:', hashlib.sha256(plan_bytes).hexdigest())
print('source:', plan['new_data_source'])
print('causality:', plan['causality_rule'])


plan sha256: f48d0ae5bd947a6f7fc4071616dc3159be5d238bb0a91e4ba3a8341a25f0b2e4
source: data/raw/external/night_session_20261010_v1
causality: Overnight futures change uses the prior 18:00 ET to current 09:30 ET window; premarket uses 04:00-09:30 ET. Both end at the cash open. The traded target is the same day's 09:30-16:00 regular session and the benchmark is SOXX over that identical window.


## 2. 运行（qlib Recorder）


In [2]:
from src.workflow import night_session_signal as workflow
from src.workflow.night_session_signal import write_card_files

summary = workflow.run_night_session_signal(root=ROOT)
write_card_files(summary, card_dir=CARD)
print('decision:', summary['decision'])
print('selected:', summary['selected_signal'])
print('promotion_eligible:', summary['promotion_eligible'])
print('quarterly wins:', summary['quarterly_win_count'], '/', summary['quarterly_count'])
print('recorder_id:', summary['recorder_id'])


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


2026/10/09 17:08:38 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[40474:MainThread](2026-10-09 17:08:38,298) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[40474:MainThread](2026-10-09 17:08:38,299) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[40474:MainThread](2026-10-09 17:08:38,299) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


[40474:MainThread](2026-10-09 17:08:38,303) WARNING - qlib.workflow - [expm.py:231] - No valid experiment found. Create a new experiment with name night_session_signal_20261010.


[40474:MainThread](2026-10-09 17:08:38,305) INFO - qlib.workflow - [exp.py:258] - Experiment 916554269708440703 starts running ...


[40474:MainThread](2026-10-09 17:08:38,361) INFO - qlib.workflow - [recorder.py:345] - Recorder 6dd90ea70bf14941b4aeece1a2857b73 starts running under Experiment 916554269708440703 ...


[40474:MainThread](2026-10-09 17:08:38,434) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


decision: stable_overnight_excess_candidate
selected: nq_overnight__cash
promotion_eligible: True
quarterly wins: 8 / 9
recorder_id: 6dd90ea70bf14941b4aeece1a2857b73


## 3. 信息系数（隔夜信号 → 当日 regular session）


In [3]:
import pandas as pd
ic = pd.read_csv(CARD / 'information_coefficients.csv')
display(ic.round(4))


,signal,ic_soxl,ic_excess,observations,classification
0,nq_overnight,0.2200,0.2244,498,tradeable_forecast
1,es_overnight,0.1381,0.1444,498,tradeable_forecast
2,soxl_premarket,-0.0292,-0.0246,499,weak_or_none
3,soxx_premarket,0.0011,0.0042,497,weak_or_none
4,smh_premarket,-0.0195,-0.0159,498,weak_or_none


## 4. 策略对比（development / diagnostic × 10/25bp）


In [4]:
comp = pd.read_csv(CARD / 'signal_comparison.csv')
sel = comp[comp.signal==summary['selected_signal']]
cols=['window','cost_bps','return_pct','benchmark_return_pct','sharpe','benchmark_sharpe','sharpe_difference','max_drawdown_improvement_pct','switches_per_year','stable_excess']
display(sel[cols].round(3))


,window,cost_bps,return_pct,benchmark_return_pct,sharpe,benchmark_sharpe,sharpe_difference,max_drawdown_improvement_pct,switches_per_year,stable_excess
1,development,10.0,157.252,-0.204,1.418,0.149,1.269,-3.398,235.144,True
11,reused_diagnostic,10.0,260.161,11.848,2.788,0.603,2.185,5.939,245.700,False
21,development,25.0,69.296,-0.204,0.940,0.149,0.791,0.361,235.144,False
31,reused_diagnostic,25.0,168.888,11.848,2.221,0.603,1.618,7.886,245.700,False


## 5. 季度一致性


In [5]:
q = pd.read_csv(CARD / 'quarterly_comparison.csv')
display(q[q.cost_bps==25.0].round(3))


,cost_bps,quarter,sessions,wealth_ratio_vs_soxx,average_exposure
9,25.0,2024Q4,58,1.070,0.483
10,25.0,2025Q1,60,1.248,0.433
11,25.0,2025Q2,62,1.472,0.613
12,25.0,2025Q3,64,1.122,0.562
13,25.0,2025Q4,64,0.740,0.594
14,25.0,2026Q1,61,1.350,0.525
15,25.0,2026Q2,62,1.708,0.645
16,25.0,2026Q3,62,1.070,0.500
17,25.0,2026Q4,6,1.016,0.667


## 6. 中文结论


In [6]:
print('选中信号:', summary['selected_signal'])
print('季度跑赢:', summary['quarterly_win_count'],'/',summary['quarterly_count'])
print('决定:', summary['decision'], '| promotion_eligible:', summary['promotion_eligible'])
print('qlib Recorder:', summary['recorder_id'])


选中信号: nq_overnight__cash
季度跑赢: 8 / 9
决定: stable_overnight_excess_candidate | promotion_eligible: True
qlib Recorder: 6dd90ea70bf14941b4aeece1a2857b73
